In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

df = pd.read_csv("clauses.csv")
print(len(df), "clauses")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

188 clauses


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [2]:
def train_for(label):
    labels = sorted(df[label].unique())
    label2id = {name: i for i, name in enumerate(labels)}

    train_df, test_df = train_test_split(
        df, test_size=0.25, random_state=42, stratify=df[label])

    def to_dataset(part):
        ds = Dataset.from_dict({
            "text": part["text"].tolist(),
            "label": [label2id[x] for x in part[label]],
        })
        return ds.map(lambda b: tokenizer(b["text"], truncation=True,
                                          padding="max_length", max_length=128),
                      batched=True)

    train_ds, test_ds = to_dataset(train_df), to_dataset(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels),
        id2label=dict(enumerate(labels)), label2id=label2id)

    args = TrainingArguments(
        output_dir=f"out_{label}",
        num_train_epochs=8,
        per_device_train_batch_size=16,
        learning_rate=5e-5,
        save_strategy="no",
        report_to="none",
        seed=42,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds)
    trainer.train()

    preds = trainer.predict(test_ds).predictions.argmax(-1)
    print(f"===== DistilBERT results for: {label} =====")
    print(classification_report(test_ds["label"], preds,
                                labels=list(range(len(labels))),
                                target_names=labels, zero_division=0))

    trainer.save_model(f"model_{label}")
    tokenizer.save_pretrained(f"model_{label}")

In [3]:
train_for("clause_type")
train_for("risk")

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


===== DistilBERT results for: risk =====
              precision    recall  f1-score   support

      Normal       0.83      0.89      0.86        27
       Risky       0.83      0.75      0.79        20

    accuracy                           0.83        47
   macro avg       0.83      0.82      0.82        47
weighted avg       0.83      0.83      0.83        47



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
!zip -r models.zip model_clause_type model_risk
from google.colab import files
files.download("models.zip")


  adding: model_clause_type/ (stored 0%)
  adding: model_clause_type/model.safetensors (deflated 8%)
  adding: model_clause_type/tokenizer.json (deflated 71%)
  adding: model_clause_type/tokenizer_config.json (deflated 43%)
  adding: model_clause_type/training_args.bin (deflated 54%)
  adding: model_clause_type/config.json (deflated 53%)
  adding: model_risk/ (stored 0%)
  adding: model_risk/model.safetensors (deflated 8%)
  adding: model_risk/tokenizer.json (deflated 71%)
  adding: model_risk/tokenizer_config.json (deflated 43%)
  adding: model_risk/training_args.bin (deflated 54%)
  adding: model_risk/config.json (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import pipeline

risk = pipeline("text-classification", model="model_risk")
ctype = pipeline("text-classification", model="model_clause_type")

tests = [
    "The landlord can keep the full deposit if the tenant leaves early.",
    "You must work for us for two years or pay back all training costs.",
    "The flat owner will fix any water leakage at his own cost.",
]
for t in tests:
    print(t)
    print("  type:", ctype(t)[0]["label"], "| risk:", risk(t)[0]["label"],
          f"({risk(t)[0]['score']:.0%})")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

The landlord can keep the full deposit if the tenant leaves early.
  type: Deposit | risk: Risky (98%)
You must work for us for two years or pay back all training costs.
  type: Salary | risk: Risky (95%)
The flat owner will fix any water leakage at his own cost.
  type: Maintenance | risk: Risky (94%)
